## Author - John P. George 
## Created on - 04 December 2025
## Description - Machine learning using SVC and Naïve Bayes for the classification of long COVID versus COVID-19, followed by AI explainability using SHAP.

In [ ]:
import pandas as  pd 
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.model_selection import cross_val_score, KFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report, f1_score, roc_curve, roc_auc_score
import shap 
from shap import KernelExplainer
from sklearn.feature_extraction.text import TfidfVectorizer #reducing features
from sklearn.feature_selection import SelectKBest #feature selection
from sklearn.inspection import permutation_importance
import joblib

In [ ]:
df = pd.read_csv("hgnc_171LC177Cov_EST-VST_103125.tsv", index_col=0)
print (df.shape)
X = df.drop("Class_label", axis=1)
y = df['Class_label']


In [ ]:
# ----- normal train-test split -----
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

scaler = StandardScaler()

# ----- Model defining -----
nb_model = GaussianNB()
svm_model = SVC(kernel='rbf', probability=True, random_state=42)

pipe = make_pipeline (scaler, svm_model)    ## Select svm_model for SVC or nb_model for Naive Bayes in the pipe

# ----- Split training into train + validation -----
X_train_cv, X_val, y_train_cv, y_val = train_test_split(X_train, y_train, test_size=0.2, stratify=y_train, random_state=42)

# ----- Evaluate on validation set -----
pipe.fit(X_train_cv, y_train_cv) 
#Testing on validation data
print("Accuracy on validation data for SVM: ", pipe.score(X_val, y_val)) 

# ----- 5-fold cross validation -----
scorespipe = cross_val_score (pipe, X_train, y_train, cv = 5)
print ("SVM - ",scorespipe)
print ("mean SVM - ",np.average(scorespipe))

In [ ]:
# ----- Train on complete training data -----

pipe.fit(X_train, y_train)
y_test_pred = pipe.predict(X_test)
#print (X_test, y_test_pred)

# -----  Model evaluation on Test data -----
test_accuracy = pipe.score(X_test, y_test)
acc = accuracy_score(y_test, y_test_pred)
print("Score on unseen test data: ", test_accuracy) 
print ("Accuracy: ", acc)

In [ ]:
# -----  Prediction and classification report -----

print (classification_report(y_test, y_test_pred))

# -----  Confusion matrix -----
cm = confusion_matrix(y_test, y_test_pred)
print(cm)

# Extract TN, FP, FN, TP
tn, fp, fn, tp = cm.ravel()

# -----  Calculate Sensitivity and Specificity -----
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)
f1 = f1_score(y_test, y_test_pred)

print(f"\nSensitivity (Recall / True Positive Rate): {sensitivity:.2f}")
print(f"Specificity (True Negative Rate): {specificity:.2f}")
print(f"f1 score : {f1:.2f}")

cm_disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=pipe.classes_)
cm_disp.plot(cmap = 'Blues')

In [ ]:
# -----  Calculate ROC  -----
y_pred_proba = pipe_nb.predict_proba(X_test)[:, 1] # Probabilities of the positive class
#print (y_pred_proba)
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
auc_score = roc_auc_score(y_test, y_pred_proba)
print(f"ROC AUC Score: {auc_score}")

# ----- ROC-AUC curve ploting -----
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {auc_score:.2f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier') # Baseline for random classification
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# ------ SHAP analysis  ---------

def predict_proba_wrapper(X_input):
    # Check if input is a DataFrame or NumPy array
    if isinstance(X_input, pd.DataFrame):
        X_df = X_input
    else:
        X_df = pd.DataFrame(X_input, columns=X.columns)
    
    # Return the probability of the positive class
    return pipe.predict_proba(X_df)[:, 1]

# 2. Select a background dataset for the explainer
# A representative sample from the training data is a good choice.
# Using a small subset speeds up the explanation process significantly.
background_data = X_train.sample(5, random_state=42) 

# 3. Initialize the KernelExplainer with the wrapper function and background data
explainer = shap.KernelExplainer(predict_proba_wrapper, background_data,    )

shap_values = explainer.shap_values(X_test) 

shap_df = pd.DataFrame(shap_values, columns=X_test.columns)

# Optionally, add the model’s predictions or true labels
shap_df['predicted_proba'] = pipe.predict_proba(X_test)[:, 1]
shap_df['true_label'] = y_test.values

# Save to CSV
shap_df.to_csv("shap_values_dataframe_110125.csv", index=False)
shap_df.to_csv("shap_values_dataframe_110125.tsv", sep = '\t', index=False)

#shap_df = pd.read_csv("/media/iob/sda1/iob/John/Transcriptomic_LC/HICs_ISGs/Old_results/shap_values_dataframe_110125.csv")


In [ ]:
# Columns that are NOT SHAP values
meta_cols = ["predicted_proba", "true_label"]

# SHAP values only
shap_values = shap_df.drop(columns=meta_cols)

# Metadata
predicted_proba = shap_df["predicted_proba"]
true_label = shap_df["true_label"]
shap_values_array = shap_values.values

In [ ]:
# Select gene columns (exclude prediction and label columns)
gene_cols = [c for c in shap_df.columns if c not in ['predicted_proba', 'true_label']]

# Create two-column summary: Gene + Mean SHAP value
mean_shap_df = pd.DataFrame({
    "Gene": gene_cols,
    "Mean_SHAP": shap_df[gene_cols].mean().values
})

# Save output
mean_shap_df.to_csv("mean_shap_per_allgenes_120125.csv", index=False)
mean_shap_df.to_csv("mean_shap_per_allgenes_120125.tsv", sep="\t", index=False)


In [ ]:
# ------- Provide genes for ion channels and immune signaling ---------

gene_list = pd.read_csv ("HIC_ISG_genelist.tsv", sep='\t') 
gene_list = gene_list['gene_list'].tolist()

gene_list_extended = [g for g in gene_list if g in shap_df.columns] + ['predicted_proba', 'true_label']

# Filter the SHAP dataframe
shap_filtered_df = shap_df[gene_list_extended]

# Save filtered SHAP values
shap_filtered_df.to_csv("shap_values_genes_of_interest.tsv", sep='\t', index=False)


In [ ]:
shap_mean_importance = shap_filtered_df.drop(['predicted_proba', 'true_label'], axis=1).abs().mean().sort_values(ascending=False)
shap_mean_importance.to_csv("mean_shap_HIC-ISG_importance_110525.tsv", sep='\t')


In [ ]:
shap_mean_importance.head(20).plot(kind='barh', figsize=(6,8))
plt.xlabel("Mean |SHAP| Value")
plt.title("Top 20 Contributing Genes")
plt.show()


In [ ]:
grouped_mean = shap_filtered_df.groupby('true_label').mean().drop(['predicted_proba'], axis=1)
grouped_mean = grouped_mean.T  # genes as rows

plt.figure(figsize=(8,10))
grouped_mean.plot(kind='barh', figsize=(5,10))
plt.title("Mean SHAP values by Class")
plt.xlabel("Mean SHAP Value")
plt.ylabel("Gene")
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---- Defining custom gene order for ion channels and immune signaling from overall SHAP resuts ----
custom_gene_order = ['ASIC4', 'CLCN6', 'GABRD', 'GJC2', 'SCN2B', 'SLC22A23', 'SLC4A1', 'SLC9A3', 'SLC9A7', 'TRPM8', 'ANO6', 
                     'TRPM2', 'ADARB1', 'ADRB2', 'APOBEC3F', 'BRS3', 'C5', 'CAPRIN1', 'CD28', 'CDK6', 'FGR', 'FOXO3', 'GNAI3', 
                     'GSK3A', 'HMGB2', 'HSPA14', 'IFITM1', 'IKBKE', 'IL37', 'INPP5D', 'LGALS2', 'MAP3K7', 'MAPK1', 'NOTCH1', 
                     'PDGFRA', 'PIK3CB', 'POLR3GL', 'PRKACG', 'PROCR', 'PTCH1', 'S100B', 'SH2D1A', 'SIRT2', 'SREBF2', 'TCF4', 
                     'TNFAIP3', 'TRIM31', 'WAS']

grouped_melted = pd.read_csv("mean_SHAP_HIC-ISG_by_class_tidy_110525.tsv", sep='\t')

plot_df = grouped_melted[grouped_melted['gene'].isin(custom_gene_order)]

# Keep only your custom order sequence
plot_df['gene'] = pd.Categorical(plot_df['gene'], categories=custom_gene_order, ordered=True)
plot_df = plot_df.sort_values('gene')

plt.figure(figsize=(10, 8))

for label in plot_df['class'].unique():
    subset = plot_df[plot_df['class'] == label]
    plt.barh(subset['gene'], subset['magnitude'], label=f"Class {label}", alpha=0.7)

plt.axvline(0, color='k', linestyle='--')
# 👇 Fix the gene order (top-to-bottom)
plt.gca().invert_yaxis()
plt.xlabel("Mean SHAP Value")
plt.ylabel("Genes")
plt.yticks(fontstyle='italic')
plt.title("Class-wise gene (HICs and ISGs) contribution")
plt.legend(title='Class')
plt.tight_layout()

plt.savefig("SHAP_summary_plot.pdf", format="pdf", bbox_inches="tight")
plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

custom_gene_order = [
    'ASIC4', 'CLCN6', 'GABRD', 'GJC2', 'SCN2B', 'SLC22A23', 'SLC4A1',
    'SLC9A3', 'SLC9A7', 'TRPM8', 'ANO6', 'TRPM2', 'ADARB1', 'ADRB2',
    'APOBEC3F', 'BRS3', 'C5', 'CAPRIN1', 'CD28', 'CDK6', 'FGR', 'FOXO3',
    'GNAI3', 'GSK3A', 'HMGB2', 'HSPA14', 'IFITM1', 'IKBKE', 'IL37',
    'INPP5D', 'LGALS2', 'MAP3K7', 'MAPK1', 'NOTCH1', 'PDGFRA',
    'PIK3CB', 'POLR3GL', 'PRKACG', 'PROCR', 'PTCH1', 'S100B',
    'SH2D1A', 'SIRT2', 'SREBF2', 'TCF4', 'TNFAIP3', 'TRIM31', 'WAS'
]

custom_gene_order = [g for g in custom_gene_order if g in grouped_mean_filtered.index]

grouped_mean_ordered = grouped_mean_filtered.loc[custom_gene_order]

# Plot heatmap
plt.figure(figsize=(8, 10))
sns.heatmap(
    grouped_mean_ordered,
    cmap='RdBu_r',
    center=0,
    linewidths=0.5
)

plt.title("Mean Contribution per Gene per Class", fontsize=14)
plt.xlabel("Class", fontsize=12)
plt.ylabel("Genes", fontsize=12)
plt.yticks(fontstyle='italic')
plt.tight_layout()

plt.savefig("SVC_mean_contribution_HM.pdf", format="pdf", bbox_inches="tight")
plt.show()
